# Day 1: Andreas's controller experiments

Explore position-to-velocity [PID](../../docs/glossary.md#pd-and-pid), velocity-to-tilt control and their [cascade](../../docs/glossary.md#cascade) using the supplied controllers and toy plants. Follow the [worksheet](../../docs/controller_practical.md) and [software guide](../../docs/day_1_software.md) for C1-C7, units, evidence and the connection to later days. The [controller guide](README.md) describes the files and commands.

These cells run simulation and offline replay only: they do not connect to a drone. Choose a new run ID when rerunning; existing results are preserved. Write a hypothesis before each parameter change.

In [ ]:
from pathlib import Path
import sys

# Jupyter may start in code/andreas rather than code/.
if (Path.cwd() / 'hover_pid.py').is_file():
    sys.path.insert(0, str(Path.cwd().parent))
elif (Path.cwd() / 'code' / 'andreas' / 'hover_pid.py').is_file():
    sys.path.insert(0, str(Path.cwd() / 'code'))
from andreas import practical, hover_pid, velocity_pid
root = Path(practical.__file__).resolve().parents[2]
outputs = root / 'day1' / 'notebook'
run_id = 'trial-01'

## C1-C3: plant, baseline and gain changes

Trace the controller classes and the three simulated command interfaces. Derive the position-loop characteristic equation for a velocity-lag plant; identify the units of each gain. Compare your predicted effect of kd with the saved responses, command limits and error during the final three seconds.

In [ ]:
for kd in (0.0, 0.6, 1.2):
    practical.main(['--kd', str(kd), '--output', str(outputs / f'{run_id}-kd-{kd}'), '--no-show'])

## C4: velocity loop and cascade

Inspect the velocity and cascade CSVs and plots from the preceding runs. Compare kv at fixed position gains. Explain which loops are closed offboard in each experiment and how a tilt toward +X maps to firmware roll/pitch using yaw and the frame conventions.

In [ ]:
practical.main(['--kv', '6', '--output', str(outputs / f'{run_id}-kv-6'), '--no-show'])

## C5-C6: disturbance, integral action and sensing noise

The bias option adds a constant velocity disturbance to the position-loop toy plant. Trace its entry point and compare PD/PID, including conditional anti-windup and recovery. Then compare noise sensitivity with paired seeds; preserve truth separately from measured inputs. Use development seeds before held-out evaluation.

In [ ]:
for ki in (0.0, 0.4):
    practical.main(['--bias', '0.08', '--ki', str(ki), '--output', str(outputs / f'{run_id}-bias-ki-{ki}'), '--no-show'])
practical.main(['--position-noise-std', '0.01', '--velocity-noise-std', '0.03', '--seed', '0', '--output', str(outputs / f'{run_id}-noise-0'), '--no-show'])

## C7: recorded data, hand checks and handover

The synthetic file below checks format only. Substitute a supplied real recording for experimental analysis. Compare computed velocity/tilt directions with recorded displacement and motion; a fixed recording cannot react to the computed commands.

The [software guide](../../docs/day_1_software.md) gives the separate motor-disabled capture command and preparation for flight. Before hardware work, cross-check the frame mapping, assigned radio, propeller/restraint and stop procedure within your four-person group, and agree together before proceeding.

For Day 2, distinguish these onboard estimates from raw sensor inputs. For Day 3, use what you measured about lag, command limits and noise sensitivity, together with the team's flight-check results; discuss whether a continuous race reference needs a different interface or feed-forward.

In [ ]:
practical.main(['--mode', 'replay', '--recording', str(root / 'examples' / 'day1-state.synthetic.csv'), '--output', str(outputs / f'{run_id}-format-replay'), '--no-show'])